# IAA Hackathon 2026 — Decision Log

This notebook documents all key decisions, assumptions, AI/human roles, methodology choices, and model evolution across the four-notebook pipeline.

## 1. Competition Overview

- **Competition:** IAA AI-Assisted Hackathon 2026, Part 1 (Kaggle)
- **Objective:** Predict expected ultimate claim cost per test row (pure risk premium)
- **Scoring:** Tweedie D² (power = 1.5)
- **Claims basis:** Net of excess, net of recoveries, excluding repudiated (declined) claims
- **Submissions:** 10 per day; select 1 final submission
- **Leaderboard target:** D² ≈ 0.12–0.13 (as of Oct 2026)

## 2. Data Sources

All data loaded as pandas DataFrames from `/data/` folder (parquet format).

| File | Rows | Cols |
|---|---|---|
| policies_motor | 979,968 | 81 |
| policies_buildings | 707,726 | 69 |
| policies_contents | 721,617 | 70 |
| test_motor | 254,986 | 78 |
| test_buildings | 184,155 | 66 |
| test_contents | 187,498 | 67 |
| claims | 162,546 | — |
| customers | 545,240 | — |
| addresses | 575,983 | — |
| weather | 2,879,915 | — |
| sample_submission | 626,639 | 2 |

## 3. Data Quality Issues and Fixes

| ID | Issue | Fix |
|---|---|---|
| R1 | Motor customer_id not zero-padded | Left-pad to 8 chars (10% rows wouldn't join) |
| R2 | Exposure formula inconsistent | Standardised to (period_end − period_start + 1) / 365 |
| R3 | Construction_type timber codes | Map both 'T' (pre-2022) and 'W' (2023+) to TIMBER |
| R4 | Duplicate claims | Removed (per DATA_CONTRACT.md) |
| R5 | compulsory_excess | Dropped — constant per line, no predictive value |
| R6 | vehicle_colour | Dropped — gender bias concern |
| R7 | Leakage features | Excluded: written_premium, customer_service_contacts, renewal_decision, discount_applied |
| R8 | PII | Excluded: surname, date_of_birth |

## 4. Key Decision Chronology

### 4.1 Claims Development to Ultimate (Notebook 1)

| # | Decision | Rationale | Made by |
|---|---|---|---|
| 1 | Use external benchmark development factors | Only 4 years of data (2021–2024); bodily injury takes 10+ years to develop — data alone cannot capture full tail | Human (pricing actuary) |
| 2 | bodily_injury only 75% developed at age 3 | External market benchmarks for long-tail liability | Human confirmed, AI implemented |
| 3 | subsidence only 82% developed at age 3 | Known long-tail peril with seasonal pattern | Human confirmed, AI implemented |
| 4 | Short-tail perils (own_damage, theft, glass_hail) 90–95%+ at age 0 | Fast-developing perils, minimal IBNR | AI proposed, human reviewed |
| 5 | Claims valued at 31 Dec 2024; dev_age = 2024 − accident_year | Standard actuarial valuation date | AI proposed, human confirmed |
| 6 | Match claims to policy rows by loss_date ∈ [period_start, period_end] | Correctly handles multiple rows per policy per year | AI |
| 7 | Allocate claims matching multiple rows by exposure weight | Pro-rata allocation prevents double-counting | AI |
| 8 | Exposure formula: (end − start + 1) / 365 | Per DATA_CONTRACT.md specification | Human specified |

### 4.2 Feature Selection Evolution (Notebooks 2 & 3)

| # | Decision | Rationale | Made by |
|---|---|---|---|
| 9 | Initial: manual feature selection via Features_v1.xlsx | First-pass approach using domain knowledge | Human |
| 10 | Dropped compulsory_excess (constant per line) | No predictive value | Human approved |
| 11 | V2: Switched to data-driven selection (ALL features minus exclusions) | Manual spreadsheet excluded entire feature groups (e.g., address property features for motor); flood_zone was top feature for buildings/contents but excluded for motor | AI proposed, human approved |
| 12 | Added address property features to motor (flood_zone, elevation_m, distance_to_river_km, soil_type, etc.) | Weather-related claims affect motor too; flood_zone proved valuable | AI |
| 13 | GLM: target encoding for high-cardinality categoricals (make_model, occupation, region, marketing_segment) | One-hot encoding 300+ levels caused convergence failure; target encoding with Bayesian shrinkage (SMOOTHING=30) is standard actuarial practice | AI |
| 14 | GLM: cardinality filter (max 10 unique values for one-hot) | Prevents one-hot blowup; high-cardinality handled by target encoding instead | AI |

### 4.3 Model Design and Tuning

| # | Decision | Rationale | Made by |
|---|---|---|---|
| 15 | XGBoost objective: reg:tweedie (power=1.5) | Matches competition scoring metric | AI |
| 16 | log(exposure) as offset (base_margin) | Standard actuarial approach — predictions proportional to exposure | AI |
| 17 | Time-based split: 2021–2023 train, 2024 validate | Simulates real-world forecasting; avoids random split leakage | Human specified |
| 18 | GLM: switched from sklearn TweedieRegressor to statsmodels GLM | sklearn lacks proper offset support (log_exposure as regularized feature distorted predictions); statsmodels IRLS is more stable | AI |
| 19 | GLM: StandardScaler on numeric features | Prevents IRLS convergence issues with features at different scales | AI |
| 20 | max_depth=6 (reverted from 8) | max_depth=8 with 64 features caused severe overfitting (D² went negative); depth 6 with stronger regularization works best | AI |
| 21 | Stronger XGBoost regularization: colsample_bytree=0.5, min_child_weight=10, reg_alpha=0.5, reg_lambda=2.0 | Needed to handle 64 features without overfitting | AI |
| 22 | Trend uplift OFF (TREND_UPLIFT = False) | Buildings CAGR of 21.3% was likely a one-off (bad storm year 2024), not sustainable; removing it lowered predictions ~10% and improved Kaggle score | AI proposed, human confirmed |
| 23 | Bias correction from validation set | Calibrates 2021–2023 model predictions to 2024 actuals | AI |

## 5. AI / Human Role Log

### Human (Pricing Actuary) Decisions
- Defined the 4-notebook structure (Data Prep → XGBoost → GLM & Blend → Decision Log)
- Specified exposure formula and construction_type mapping
- Confirmed bodily injury can take 10+ years to develop — external benchmarks required
- Approved dropping compulsory_excess (constant per line)
- Specified time-based validation (2021–2023 train, 2024 validate)
- Requested pandas over Spark for simplicity and reproducibility
- Approved data-driven feature selection replacing manual spreadsheet
- Confirmed trend uplift should be disabled (buildings CAGR 21.3% likely one-off)
- Requested Excel checkpoints at each stage for audit trail
- Requested explicit logging of AI/human roles, assumptions, and decisions

### AI (Genie Code) Contributions
- Generated all code across 4 notebooks
- Researched and proposed external benchmark development factors by peril
- Implemented claims matching to policy rows (loss_date within period)
- Built claims allocation logic for multi-row matches (exposure-weighted)
- Implemented XGBoost Tweedie with base_margin offset
- Built GLM with statsmodels (after initial sklearn approach failed to converge)
- Implemented target encoding with Bayesian shrinkage for high-cardinality categoricals
- Designed and implemented cardinality filtering for GLM feature selection
- Diagnosed and fixed GLM convergence issues (3 iterations of fixes)
- Diagnosed XGBoost overfitting with max_depth=8 and increased regularization
- Generated all Excel checkpoint files
- Produced all submission CSVs
- Wrote this decision log

## 6. Model Comparison (Final)

### Validation D² Scores (2024 holdout)

| Line | XGBoost D² | GLM D² | Best Model |
|---|---|---|---|
| Motor | 0.0490 | 0.0470 | XGBoost (marginal) |
| Buildings | 0.0562 | **0.0806** | GLM |
| Contents | 0.0517 | **0.0729** | GLM |
| **Average** | 0.0523 | **0.0668** | **GLM** |

### XGBoost Configuration

| Parameter | Value |
|---|---|
| objective | reg:tweedie (power=1.5) |
| max_depth | 6 |
| learning_rate | 0.05 |
| subsample | 0.8 |
| colsample_bytree | 0.5 |
| min_child_weight | 10 |
| reg_alpha | 0.5 |
| reg_lambda | 2.0 |
| num_boost_round | 500 (early stopping patience=50) |
| Features (motor) | 64 (data-driven, all minus exclusions) |
| Offset | log(exposure) as base_margin |
| Bias correction | Validation mean ratio |
| Trend uplift | OFF |

### GLM Configuration

| Parameter | Value |
|---|---|
| Engine | statsmodels GLM (IRLS) |
| Family | Tweedie (var_power=1.5, log link) |
| Offset | log(exposure) — proper actuarial offset |
| Feature source | XGBoost top 30 importance |
| Feature filter | Cardinality ≤ 10 for one-hot; target encoding for high-cardinality |
| MAX_FEATS | 15 per line |
| Target encoding smoothing | 30 (Bayesian shrinkage) |
| Numeric scaling | StandardScaler |
| Bias correction | Validation mean ratio |
| Trend uplift | OFF |

### GLM Selected Features by Line

**Motor (20 cols):** Year, ncb_years, rainfall_mm, driver_age, vehicle_use, urban_density, years_licensed, make_model_te, annual_mileage, years_at_address, marketing_segment_te, prior_claims_5y, named_young_driver, occupation_te, mean_temperature_c, channel

**Buildings (19 cols):** flood_zone, elevation_m, sum_insured, distance_to_river_km, occupancy, missed_payments_12m, prior_claims_5y, Year, trees_within_5m, region_te, channel, year_built, listed_building, bedrooms, credit_band

**Contents (18 cols):** flood_zone, elevation_m, sum_insured, accidental_damage_cover, prior_claims_5y, distance_to_river_km, urban_density, tenure, payment_method, channel, missed_payments_12m, voluntary_excess, high_value_items, Year, policyholder_age

## 7. Key Assumptions and Caveats

1. **Development factors are external benchmarks**, not derived from the data (only 4 years of history available). They represent developed-market insurance patterns adapted for this synthetic portfolio.
2. **Trend uplift is OFF.** The 2021–2024 data shows strong upward trend (buildings +78%), but this is likely a one-off effect (e.g., a particularly bad storm year in 2024) rather than a sustainable CAGR. Bias correction alone calibrates to the validation period.
3. **Bias correction assumes 2024 is representative of 2025.** If 2025 has different claims experience, the bias factor may under- or over-correct.
4. **Target encoding uses training split only (2021–2023)** to avoid leakage. Unseen categories in test data fall back to overall mean.
5. **GLM uses fewer features than XGBoost** (15–20 vs 53–64) but with proper offset and target encoding. The GLM outperforms XGBoost on buildings and contents.
6. **All code uses pandas** (not Spark) per user preference for simplicity and reproducibility.
7. **No hyperparameter tuning** beyond manual adjustments — XGBoost uses sensible defaults with stronger regularization for the larger feature set.
8. **Weather data for 2025** uses the latest available year (2024) as a proxy.

## 8. Submission History

| Version | File | Mean pred | Notes |
|---|---|---|---|
| v1 (initial) | xgboost_tweedie_v1.csv | 479.88 | 42 features, trend uplift ON, max_depth=6 |
| v1 (initial) | blend_5050_v1.csv | 519.61 | 50/50 XGBoost+GLM, GLM was broken (D²≈0) |
| v1 (initial) | blend_7030_v1.csv | 503.72 | 70/30 XGBoost-weighted |
| v2 (GLM fixed) | glm_tweedie_v1.csv | 559.34 | statsmodels GLM, target encoding, trend ON |
| v2 (GLM fixed) | blend_5050_v1.csv | 519.61 | GLM now strong but trend inflated predictions |
| v3 (current) | xgboost_tweedie_v1.csv | 433.93 | 64 features, trend OFF, stronger regularization |
| v3 (current) | blend_5050_v1.csv | 466.44 | **Recommended first submission** |
| v3 (current) | blend_7030_v1.csv | 453.44 | More conservative option |
| v3 (current) | glm_tweedie_v1.csv | 498.96 | GLM-only, highest predictions |

## 9. Files Produced

### Notebooks (`notebooks_v3/`)
- `01 Data Prep and Claims` — Data loading, cleaning, claims development, enrichment, save to parquet
- `02 XGBoost Tweedie` — Data-driven feature selection, XGBoost training, prediction, submission
- `03 GLM and Blend` — statsmodels GLM with target encoding, blend with XGBoost, submission
- `04 Decision Log` — This notebook

### Processed Data (`data/proc/`)
- `train_{motor,buildings,contents}.parquet` — Processed training data with ultimate_claims
- `test_{motor,buildings,contents}.parquet` — Processed test data
- `sample_submission.parquet` — Submission template

### Excel Checkpoints (`Excel files_v3/`)
- `claims_development_checkpoint.xlsx` — Development factors, claims by peril, trend analysis
- `xgboost_model_checkpoint.xlsx` — XGBoost metrics, top 30 feature importance per line
- `glm_blend_checkpoint.xlsx` — GLM metrics, model comparison, blend summary

### Submissions (`submissions_v3/`)
- `xgboost_tweedie_v1.csv` — XGBoost only
- `glm_tweedie_v1.csv` — GLM only
- `blend_5050_v1.csv` — 50/50 blend
- `blend_7030_v1.csv` — 70/30 XGBoost-weighted blend